# Unified Word Training (ASL + ArSL, Independent Vocab)

**How to use**

1. Set `LANGUAGE = "asl"` or `LANGUAGE = "arsl"`
2. Update paths in **Cell 2** (`PROJECT_ROOT`)
3. Ensure the vocab CSV for each language exists
4. Run all cells top-to-bottom


## Cell 1: Imports


In [1]:
import os
import json
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
import mediapipe as mp

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    BatchNormalization,
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("✅ Imports loaded")
print("TensorFlow:", tf.__version__)


✅ Imports loaded
TensorFlow: 2.10.0


## Cell 2: Global config


In [2]:
# =========================
# CELL 2: KAGGLE CONFIGURATION
# =========================
from pathlib import Path
import json

LANGUAGE = "arsl"
DATASET_TYPE = "folder_classid"
FEATURES_PER_FRAME = 258
SEQUENCE_LENGTH = 30

VIDEOS_DIR = Path("/kaggle/input/your-dataset/videos_or_npys") 
WORK_DIR = Path("/kaggle/working")
VOCAB_CSV = Path("/kaggle/input/karsl502-vocab/KARSL-502_Labels.xlsx")

if not WORK_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)

version = 1
while True:
    test_path = WORK_DIR / f"{LANGUAGE}_word_lstm_model_best_v{version}.h5"
    if not test_path.exists(): break
    version += 1

MODEL_BEST = WORK_DIR / f"{LANGUAGE}_word_lstm_model_best_v{version}.h5"
MODEL_FINAL = WORK_DIR / f"{LANGUAGE}_word_lstm_model_final_v{version}.h5"
CACHE_NPZ = WORK_DIR / f"{LANGUAGE}_word_sequences.npz"
CLASSES_CSV = WORK_DIR / f"{LANGUAGE}_word_classes.csv"
SCALER_STATS = WORK_DIR / f"{LANGUAGE}_scaler_stats.npz"

print(f"🌟 KAGGLE RUN CONFIG - {LANGUAGE.upper()} VERSION v{version} 🌟")


Launching Configuration Panel...
🌟 RUN CONFIG - TRAINING VERSION v2 🌟
Language     : ARSL
Features     : 258
Video Source : E:\Downloads\Arabic Words Dataset
Workspace    : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)
Output Best  : arsl_word_lstm_model_best_v2.h5
Output Final : arsl_word_lstm_model_final_v2.h5


## Cell 3: GPU setup


In [3]:
print("=" * 60)
print("GPU SETUP")
print("=" * 60)
gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print("✅ Memory growth enabled")
    except RuntimeError as e:
        print("⚠️ GPU setup warning:", e)
else:
    print("⚠️ No GPU detected; CPU mode.")


GPU SETUP
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Memory growth enabled


## Cell 4: Load language vocab (auto-generate from shared if missing)


In [4]:
# =========================
# CELL 4: LOAD LANGUAGE VOCAB (BULLETPROOF PARSER)
# =========================
import pandas as pd

# 1. If you selected a file in the popup:
if VOCAB_CSV is not None and str(VOCAB_CSV).strip() != "" and VOCAB_CSV.exists():
    print(f"📄 Loading Vocab from file: {VOCAB_CSV}")
    
    # Read it as pure text first so Python doesn't crash on headers
    if str(VOCAB_CSV).lower().endswith(('.xlsx', '.xls')):
        vocab = pd.read_excel(VOCAB_CSV, header=None, dtype=str)
    else:
        vocab = pd.read_csv(VOCAB_CSV, encoding='utf-8-sig', header=None, dtype=str)
    
    # Force the first column to be ID and second to be Label
    vocab.rename(columns={0: 'source_class_id', 1: 'label_name'}, inplace=True)
    
    # Check if the very first row is text (like 'SignID') instead of a number (like '0')
    first_val = str(vocab['source_class_id'].iloc[0]).strip()
    if not first_val.isdigit():
        print(f"⚠️ Detected header row '{first_val}'. Skipping first row...")
        vocab = vocab.iloc[1:].reset_index(drop=True) # Drop the header row

    # Now it is completely safe to convert the column to integers!
    vocab["label_name"] = vocab["label_name"].astype(str).str.strip()
    vocab["source_class_id"] = vocab["source_class_id"].astype(int)

    allowed_class_ids = set(vocab["source_class_id"].tolist())
    classid_to_label = dict(zip(vocab["source_class_id"], vocab["label_name"]))
    print(f"✅ Loaded vocab rows: {len(vocab)}")
    print(f"✅ Allowed classes  : {len(allowed_class_ids)}")

# 2. If you left the CSV blank (Auto-Detect Mode):
else:
    print("⚠️ No Vocab CSV provided. Auto-generating classes directly from folder names...")
    allowed_class_ids = set()
    classid_to_label = {}
    
    if DATASET_TYPE == "folder_classid" and VIDEOS_DIR and VIDEOS_DIR.exists():
        # Look inside the Video Directory for numbered folders (0, 1, 2...)
        for folder in VIDEOS_DIR.iterdir():
            if folder.is_dir() and folder.name.isdigit():
                class_id = int(folder.name)
                allowed_class_ids.add(class_id)
                classid_to_label[class_id] = str(class_id) 
                
        print(f"✅ Auto-detected {len(allowed_class_ids)} classes from your dataset folders.")
        
        if len(allowed_class_ids) == 0:
            raise ValueError("❌ Found no numbered folders (0, 1, 2...) in your Video Dataset Folder!")
    else:
        raise ValueError("❌ No CSV provided AND Dataset Type is not 'folder_classid'. Cannot build vocabulary.")


📄 Loading Vocab from file: M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\KARSL-502_Labels.xlsx
⚠️ Detected header row 'SignID'. Skipping first row...
✅ Loaded vocab rows: 502
✅ Allowed classes  : 502


## Cell 5: MediaPipe helpers


In [5]:
mp_holistic = mp.solutions.holistic

def extract_tier3_keypoints(frame_bgr, holistic_obj):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = holistic_obj.process(frame_rgb)
    
    # 1. Pose: 33 x 4 = 132 features
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten()
    else:
        pose = np.zeros(132, dtype=np.float32)

    # 2. Left hand: 21 x 3 = 63 features
    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
        has_lh = True
    else:
        lh = np.zeros(63, dtype=np.float32)
        has_lh = False

    # 3. Right hand: 21 x 3 = 63 features
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
        has_rh = True
    else:
        rh = np.zeros(63, dtype=np.float32)
        has_rh = False

    # Combine into 258 features
    vec = np.concatenate([pose, lh, rh])
    
    # We consider a hand "detected" if either left or right hand is found
    has_hand = has_lh or has_rh
    
    return vec, has_hand

def to_fixed_sequence(seq, seq_len=30, feat_dim=258):
    n = len(seq)
    if n == 0:
        return np.zeros((seq_len, feat_dim), dtype=np.float32)
    if n >= seq_len:
        idx = np.linspace(0, n - 1, seq_len).astype(int)
        return np.array([seq[i] for i in idx], dtype=np.float32)
    out = np.zeros((seq_len, feat_dim), dtype=np.float32)
    out[:n] = np.array(seq, dtype=np.float32)
    return out


## Cell 6: Build sample list


In [6]:
samples = []
if DATASET_TYPE == "wlasl":
    wlasl_json = "/kaggle/input/wlasl-dataset/WLASL_v0.3.json"
    try:
        with open(wlasl_json, "r", encoding="utf-8") as f: data = json.load(f)
        for idx, item in enumerate(data):
            class_id = item.get("gloss_id", item.get("class_id", item.get("id", idx)))
            if class_id is None: continue
            try: class_id = int(class_id)
            except: continue
            if class_id not in allowed_class_ids: continue
            for inst in item.get("instances", []):
                vid = inst.get("video_id", inst.get("id", None))
                if vid is None: continue
                vp = VIDEOS_DIR / f"{vid}.mp4"
                if not vp.exists():
                    vp2 = VIDEOS_DIR / f"{vid}.npy"
                    if vp2.exists(): vp = vp2
                samples.append({"video_path": vp, "class_id": class_id, "label_name": classid_to_label[class_id]})
    except FileNotFoundError:
        print(f"⚠️ Could not find {wlasl_json}")
elif DATASET_TYPE == "folder_classid":
    for ext in ["*.mp4", "*.npy"]:
        for vp in VIDEOS_DIR.rglob(ext):
            try: class_id = int(vp.parent.name)
            except ValueError: continue
            if class_id not in allowed_class_ids: continue
            samples.append({"video_path": vp, "class_id": class_id, "label_name": classid_to_label[class_id]})
print(f"✅ Indexed samples: {len(samples)}")


🔍 Deep searching for videos in: E:\Downloads\Arabic Words Dataset
✅ Indexed samples: 9564


## Cell 7: Extract or load cache


In [7]:
USE_CACHE_IF_EXISTS = False
if USE_CACHE_IF_EXISTS and CACHE_NPZ.exists():
    z = np.load(CACHE_NPZ, allow_pickle=True)
    print("Loaded cache")
    X = z["X"]
    y_text = z["y_text"] if "y_text" in z else z["y"]
    print("X:", X.shape, "| y:", y_text.shape)
else:
    X_list, y_list = [], []
    with mp_holistic.Holistic(static_image_mode=False, model_complexity=1) as holistic_obj:
        for s in tqdm(samples, desc=f"Extracting ({LANGUAGE})স্থিতي"):
            vp = s["video_path"]
            if not vp.exists(): continue
            if vp.suffix.lower() == ".npy":
                try:
                    seq = np.load(str(vp))
                    if len(seq.shape) == 2 and seq.shape[1] == FEATURES_PER_FRAME:
                        seq_fixed = to_fixed_sequence(seq, SEQUENCE_LENGTH, FEATURES_PER_FRAME)
                        X_list.append(seq_fixed)
                        y_list.append(s["label_name"])
                except Exception: pass
                continue
            cap = cv2.VideoCapture(str(vp))
            seq = []
            total_frames = 0
            detected_frames = 0
            while True:
                ok, frame = cap.read()
                if not ok: break
                total_frames += 1
                vec, has_hand = extract_tier3_keypoints(frame, holistic_obj)
                if has_hand: detected_frames += 1
                seq.append(vec)
            cap.release()
            if total_frames == 0 or detected_frames / total_frames < 0.2: continue
            seq_fixed = to_fixed_sequence(seq, SEQUENCE_LENGTH, FEATURES_PER_FRAME)
            X_list.append(seq_fixed)
            y_list.append(s["label_name"])
    X = np.array(X_list, dtype=np.float32)
    y_text = np.array(y_list)
    np.savez_compressed(CACHE_NPZ, X=X, y_text=y_text)
if len(X) == 0: raise RuntimeError("No extracted samples found.")


Extracting (arsl):  79%|███████▉  | 7555/9564 [7:55:01<2:06:19,  3.77s/it] 

KeyboardInterrupt



## Cell 8: Preprocess + split


In [ ]:
N, T, F = X.shape
X2 = X.reshape(-1, F)

scaler = StandardScaler()
X2 = scaler.fit_transform(X2)
X_scaled = X2.reshape(N, T, F).astype(np.float32)

np.savez_compressed(
    SCALER_STATS,
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
)

le = LabelEncoder()
y_idx = le.fit_transform(y_text)
y_onehot = to_categorical(y_idx)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_scaled, y_onehot, test_size=0.4, random_state=42, stratify=y_idx
)
y_tmp_idx = np.argmax(y_tmp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)


print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# Save per-language class mapping
classes_df = pd.DataFrame(
    {"model_class_index": np.arange(len(le.classes_)), "label_name": le.classes_}
)
label_to_src = dict(zip(vocab["label_name"], vocab["source_class_id"]))
classes_df["source_class_id"] = classes_df["label_name"].map(label_to_src)

# Keep word_id if present (makes output compatible with the existing Live Test notebooks/scripts)
if "word_id" in vocab.columns:
    label_to_word = dict(zip(vocab["label_name"], vocab["word_id"]))
    classes_df["word_id"] = classes_df["label_name"].map(label_to_word)

classes_df.to_csv(CLASSES_CSV, index=False)
print("✅ Saved classes:", CLASSES_CSV)
display(classes_df.head())


## Cell 9: Build model


In [ ]:
num_classes = y_train.shape[1]

model = Sequential(
    [
        Bidirectional(
            LSTM(128, return_sequences=True),
            input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME),
        ),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(num_classes, activation="softmax", dtype="float32"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_acc"),
    ],
)
model.summary()


## Cell 10: Train


In [ ]:
callbacks = [
    ModelCheckpoint(
        str(MODEL_BEST),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

model.save(MODEL_FINAL)
print("✅ Saved best :", MODEL_BEST)
print("✅ Saved final:", MODEL_FINAL)


## Cell 11: Evaluation


In [ ]:
loss, acc, top5 = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}")
print(f"Test acc : {acc:.4f}")
print(f"Top-5 acc: {top5:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=np.arange(len(le.classes_)), target_names=le.classes_, zero_division=0))

print("Confusion matrix shape:", confusion_matrix(y_true, y_pred).shape)


## Cell 12: Summary


In [ ]:
print("\n✅ Done.")
print(f"Language: {LANGUAGE}")
print("Unified training summary:")
print("- Separate model/cache/classes per language")
print(
    "- If per-language vocab CSV is missing, it is generated from shared_word_vocabulary.csv"
)
print("- No merge notebook required")
